In [ ]:
# ==========================================================
#                ПОЛНЫЙ ML PIPELINE
#        (ETL → БД → ФИЧИ → TRAIN → SAVE)
# ==========================================================

import sqlite3
import pandas as pd
import json
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler


# ==========================================================
# 1️⃣ БЛОК: НАСТРОЙКИ ПРОЕКТА
# ==========================================================

DB_NAME = "my_database.db"
MODEL_PATH = "best_model.pkl"
LOG_FILE = "pipeline_log.txt"


# ==========================================================
# 2️⃣ БЛОК: ЛОГИРОВАНИЕ
# ==========================================================

def log(message):
    """Записывает сообщения в лог-файл"""
    with open(LOG_FILE, "a") as f:
        f.write(message + "\n")
    print(message)


# ==========================================================
# 3️⃣ БЛОК: СОЗДАНИЕ БАЗЫ
# ==========================================================

def create_database():
    """
    Создаём таблицу users, если её нет.
    Если структура у тебя другая — меняй SQL ниже.
    """
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
    """)

    conn.commit()
    conn.close()
    log("База данных готова")


# ==========================================================
# 4️⃣ БЛОК: EXTRACT (загрузка данных)
# ==========================================================

def extract_data():
    """
    Здесь ты можешь:
    - читать CSV
    - читать JSON
    - читать Excel
    - подключаться к API
    """

    # ПРИМЕР CSV
    df_csv = pd.DataFrame({
        "id": [1, 2],
        "name": ["Alice", "Bob"],
        "age": [25, 30]
    })

    # ПРИМЕР JSON
    json_str = '[{"id":3,"name":"Charlie","age":28}]'
    df_json = pd.DataFrame(json.loads(json_str))

    # объединяем всё
    df = pd.concat([df_csv, df_json], ignore_index=True)

    log("Данные извлечены")
    return df


# ==========================================================
# 5️⃣ БЛОК: TRANSFORM (очистка)
# ==========================================================

def transform_data(df):
    """
    Здесь делаем:
    - удаление дублей
    - заполнение пропусков
    - изменение типов
    """

    df = df.drop_duplicates(subset=["id"])
    df["age"] = df["age"].fillna(df["age"].mean())
    df["age"] = df["age"].astype(int)

    log("Данные очищены")
    return df


# ==========================================================
# 6️⃣ БЛОК: LOAD (загрузка в БД)
# ==========================================================

def load_to_db(df):
    """
    Загружаем данные в базу.
    Если id уже есть — обновляем.
    """

    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    rows_loaded = 0

    for _, row in df.iterrows():
        cur.execute("""
        INSERT INTO users (id, name, age)
        VALUES (?, ?, ?)
        ON CONFLICT(id) DO UPDATE SET
            name=excluded.name,
            age=excluded.age
        """, (row["id"], row["name"], row["age"]))

        rows_loaded += 1

    conn.commit()
    conn.close()

    log(f"{rows_loaded} строк загружено в БД")


# ==========================================================
# 7️⃣ БЛОК: FEATURE ENGINEERING
# ==========================================================

def make_features(df):
    """
    Создаём признаки для модели.
    Тут можно:
    - делать OneHotEncoding
    - масштабирование
    - создавать новые колонки
    """

    # фиктивная цель (в реальном проекте это будет реальный таргет)
    df["income"] = df["age"] * 2000  # пример зависимости

    X = df[["age"]]
    y = df["income"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    log("Фичи подготовлены")
    return X_scaled, y


# ==========================================================
# 8️⃣ БЛОК: TRAIN + EVALUATE
# ==========================================================

def train_and_select_model(X, y):
    """
    Обучаем несколько моделей и выбираем лучшую.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = {
        "Linear": LinearRegression(),
        "Tree": DecisionTreeRegressor(),
        "Forest": RandomForestRegressor()
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        results[name] = mse
        log(f"{name} MSE: {mse}")

    best_name = min(results, key=results.get)
    best_model = models[best_name]

    log(f"Лучшая модель: {best_name}")
    return best_model


# ==========================================================
# 9️⃣ БЛОК: СОХРАНЕНИЕ МОДЕЛИ
# ==========================================================

def save_model(model):
    joblib.dump(model, MODEL_PATH)
    log("Модель сохранена")


# ==========================================================
# 🔟 ГЛАВНЫЙ PIPELINE
# ==========================================================

def run_pipeline():
    """
    ЭТА ФУНКЦИЯ ЗАПУСКАЕТ ВСЁ.
    Никаких принтов ради вида.
    Всё реально выполняется.
    """

    try:
        log("===== PIPELINE START =====")

        create_database()

        df = extract_data()

        df = transform_data(df)

        load_to_db(df)

        X, y = make_features(df)

        best_model = train_and_select_model(X, y)

        save_model(best_model)

        log("===== PIPELINE FINISHED =====")

    except Exception as e:
        log("ОШИБКА: " + str(e))


# ==========================================================
# 🚀 ТОЧКА ВХОДА
# ==========================================================

if __name__ == "__main__":
    run_pipeline()
